In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bohn2016comprehension")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Bohn_2016_comprehension_tab_Study_Design_Data_Raw_Data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)


df['study_id']="bohn2016comprehension"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"subject": "ape",
                        "group.1": "group_original"})
df[['subgroup','group1']] = df['group'].str.split('-',expand=True)
# df.columns

In [3]:
df[['day', 'month', 'year']] = df['date'].str.split('.', expand=True)
df['year'] = '20' + df['year'].astype(str)

# print(df['year'])

In [4]:
df['ape'] = df['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)


comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

df = df.rename(columns={"species_y": "species",
    "sum c2 ": "sum_c2"})
df.columns = df.columns.str.replace(' ', '_')
df.dropna(subset=['ape'], inplace=True)
# df.columns

df.rename(columns={"ape": "participant"}, inplace=True)

In [5]:
remdf=[1, 3]
df = df[~df.phase.isin(remdf)]

df = df.assign(experiment_name=np.nan)
df.loc[df.session == 1, [ 'experiment_name']] =  '1a'
df.loc[df.session == 2, [ 'experiment_name']] =  '1a'

df['experiment_name'].replace( np.nan,'1b', inplace=True)

df = df.rename(columns={"condition": "condition_original",
                        "age":"age_original",
                        "subgroup":"species_subgroup"})
df = df.assign(condition_codes=np.nan)
df.loc[df.condition_original == 1, [ 'condition']] =  'arbitrary'
df.loc[df.condition_original == 2, [ 'condition']] =  'iconic'


In [6]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left') #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365

In [7]:
bohn2016comprehension_standardized=df[['study_id','experiment_name', 'year', 'month','day',
        
        'participant', 'age_original','age_in_years',
        'sex', 'species',  'species_subgroup',  'session','trial','condition',
          'app_left', 'app_right',
       'side_indicated', 'cho2',   'code2']]

In [8]:
comp_out_path_stand = os.path.join(out_pathway, 'bohn2016comprehension_exp1_standardized.csv')
bohn2016comprehension_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [9]:
names =bohn2016comprehension_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
bohn2016comprehension_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'bohn2016comprehension_exp1_glossary.csv')
bohn2016comprehension_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)